# 02 — Run MDMp hill-climbing on all datasets

Fit `mdmp.MDM(method="hc")` on every simulated dataset and write tidy CSVs
for comparison with R.

**Pinned hyperparameters:**
- `nbf = 15`
- `delta = np.arange(0.5, 1.01, 0.01)` (51 values, 0.5–1.0)
- orientation: row = parent, col = child

In [1]:
import os
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath("../.."))
from mdmp import MDM

from helpers import (
    DAGS,
    DELTA,
    METRIC_NAMES,
    NBF,
    N_REPLICATIONS,
    T,
    V,
    W,
    adjacency_to_long,
    compute_metrics,
    data_filename,
    load_true_adj,
)

DATA_DIR = "data/"

D:\dev\phd\sandbox\mdmr-python\mdmp\.venv-win\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adjacency_rows = []
metric_rows = []

true_adj_cache = {dag: load_true_adj(DATA_DIR, dag, W, V, T) for dag in DAGS}

for dag in DAGS:
    true_adj = true_adj_cache[dag]
    n_jobs = -1 if dag == "5var" else None

    for dataset_id in tqdm(range(1, N_REPLICATIONS + 1), desc=f"mdmp {dag}"):
        data_path = os.path.join(DATA_DIR, data_filename(dag, dataset_id, W, V, T))
        data = pd.read_csv(data_path)

        t0 = time.perf_counter()
        model = MDM(
            data,
            method="hc",
            nbf=NBF,
            delta=DELTA,
            verbose=False,
            n_jobs=n_jobs,
        )
        elapsed = time.perf_counter() - t0

        adj_mat = model.adj_mat.astype(int)
        metrics = compute_metrics(true_adj, adj_mat)
        n_edges = int(adj_mat.sum())

        adjacency_rows.extend(adjacency_to_long(dag, dataset_id, adj_mat))

        for metric_name in METRIC_NAMES:
            if metric_name == "computation_time":
                value = elapsed
            elif metric_name == "n_edges":
                value = n_edges
            else:
                value = metrics[metric_name]
            metric_rows.append(
                {
                    "dag": dag,
                    "dataset_id": dataset_id,
                    "metric": metric_name,
                    "value": value,
                }
            )

python_adjacency = pd.DataFrame(adjacency_rows)
python_metrics = pd.DataFrame(metric_rows)

python_adjacency.to_csv(os.path.join(DATA_DIR, "python_adjacency.csv"), index=False)
python_metrics.to_csv(os.path.join(DATA_DIR, "python_metrics.csv"), index=False)

print(f"Wrote {len(python_adjacency)} adjacency rows")
print(f"Wrote {len(python_metrics)} metric rows")

mdmp 3var:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 1/1000000 [00:00<242:48:18,  1.14it/s]

  0%|          | 2/1000000 [00:01<145:57:34,  1.90it/s]

mdmp 3var:   0%|          | 1/300 [00:04<22:21,  4.49s/it]

...

  0%|          | 4/1000000 [00:04<218:36:18,  1.27it/s]

  0%|          | 4/1000000 [00:05<353:20:41,  1.27s/it]

mdmp 5var: 100%|██████████| 300/300 [1:33:21<00:00, 18.56s/it]

mdmp 5var: 100%|██████████| 300/300 [1:33:21<00:00, 18.67s/it]

Wrote 10200 adjacency rows
Wrote 4800 metric rows
